# 📘 skimage 필터와 형태학

scikit-image의 `filters`와 `morphology` 모듈은 이미지 필터링과
형태학적 연산을 제공합니다. 과학적 이미지 처리에 특히 강력합니다.

**학습 목표:**
- 에지 검출 (Sobel, Canny, Scharr)
- 임계값 처리 (Otsu, 적응형)
- 형태학적 연산 (침식, 팽창, 열기, 닫기)
- 노이즈 제거 필터

## 1. 에지 검출

scikit-image는 다양한 에지 검출 알고리즘을 제공합니다.
OpenCV와 비슷하지만 API가 더 직관적입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  에지 검출                               │
# │  Sobel, Scharr, Prewitt, Canny           │
# └─────────────────────────────────────────┘

from skimage import data, filters, feature
import matplotlib.pyplot as plt
import numpy as np

img = data.coins()  # 흑백 동전 이미지

# 에지 검출
edges_sobel = filters.sobel(img)          # Sobel 필터
edges_scharr = filters.scharr(img)        # Scharr 필터
edges_prewitt = filters.prewitt(img)       # Prewitt 필터
edges_canny = feature.canny(img, sigma=1)  # Canny 에지
edges_canny2 = feature.canny(img, sigma=2)  # Canny (더 부드럽게)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(img, cmap='gray'); axes[0, 0].set_title('원본 (coins)')
axes[0, 1].imshow(edges_sobel, cmap='gray'); axes[0, 1].set_title('Sobel')
axes[0, 2].imshow(edges_scharr, cmap='gray'); axes[0, 2].set_title('Scharr')
axes[1, 0].imshow(edges_prewitt, cmap='gray'); axes[1, 0].set_title('Prewitt')
axes[1, 1].imshow(edges_canny, cmap='gray'); axes[1, 1].set_title('Canny (sigma=1)')
axes[1, 2].imshow(edges_canny2, cmap='gray'); axes[1, 2].set_title('Canny (sigma=2)')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 Sobel/Scharr/Prewitt: 기울기 기반 에지 검출 (float 결과)')
print('💡 Canny: 다단계 에지 검출 (이진 결과, sigma로 노이즈 조절)')
print('💡 sigma가 클수록 에지가 부드러워짐 (노이즈에 강함)')

## 2. 임계값 처리

이미지를 전경과 배경으로 분리하는 임계값 처리입니다.
Otsu, 적응형, 국소 임계값 등 다양한 방법을 제공합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  임계값 처리                              │
# │  Otsu, 적응형, 국소 임계값                │
# └─────────────────────────────────────────┘

from skimage import data, filters
from skimage.filters import threshold_otsu, threshold_local, threshold_yen
img = data.page()  # 흑백 문서 이미지

# 1. Otsu 임계값
thresh_otsu = threshold_otsu(img)
binary_otsu = img > thresh_otsu

# 2. Yen 임계값
thresh_yen = threshold_yen(img)
binary_yen = img > thresh_yen

# 3. 적응형 (국소) 임계값
block_size = 35
thresh_local = threshold_local(img, block_size, method='gaussian')
binary_local = img > thresh_local

print(f'Otsu 임계값: {thresh_otsu}')
print(f'Yen 임계값: {thresh_yen}')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(img, cmap='gray'); axes[0, 0].set_title('원본 (page)')
axes[0, 1].imshow(binary_otsu, cmap='gray'); axes[0, 1].set_title(f'Otsu ({thresh_otsu:.0f})')
axes[0, 2].imshow(binary_yen, cmap='gray'); axes[0, 2].set_title(f'Yen ({thresh_yen:.0f})')
axes[1, 0].imshow(binary_local, cmap='gray'); axes[1, 0].set_title('적응형 (gaussian)')

# 히스토그램 + 임계값
axes[1, 1].hist(img.ravel(), bins=256, color='gray', alpha=0.7)
axes[1, 1].axvline(thresh_otsu, color='red', label=f'Otsu={thresh_otsu:.0f}')
axes[1, 1].axvline(thresh_yen, color='blue', label=f'Yen={thresh_yen:.0f}')
axes[1, 1].set_title('히스토그램 + 임계값')
axes[1, 1].legend()
axes[1, 2].axis('off')
plt.tight_layout()
plt.show()

print('💡 Otsu: 히스토그램 기반 자동 임계값')
print('💡 Yen: Otsu와 비슷하지만 다른 최적화 기준')
print('💡 threshold_local: 국소적으로 적응형 임계값 (조명 불균일 시 유용)')

## 3. 형태학적 연산

이진화된 이미지의 형태를 다듬는 연산입니다.
노이즈 제거, 구멍 채우기, 객체 분리 등에 사용합니다.

| 연산 | 효과 |
|------|------|
| erosion | 침식 (객체 축소, 노이즈 제거) |
| dilation | 팽창 (객체 확대, 구멍 채우기) |
| opening | 침식→팽창 (노이즈 제거) |
| closing | 팽창→침식 (구멍 채우기) |
| white_tophat | 원본 - opening (밝은 세부) |
| black_tophat | closing - 원본 (어두운 세부) |

In [ ]:
# ┌─────────────────────────────────────────┐
# │  형태학적 연산                            │
# │  erosion, dilation, opening, closing     │
# └─────────────────────────────────────────┘

from skimage import morphology, data
from skimage.util import img_as_ubyte

# 테스트 이미지 (동전)
img = data.coins()
thresh = threshold_otsu(img)
binary = img > thresh

# 형태학적 연산
selem = morphology.disk(3)  # 원형 구조 요소 (반경 3)

eroded = morphology.erosion(binary, selem)
dilated = morphology.dilation(binary, selem)
opened = morphology.opening(binary, selem)
closed = morphology.closing(binary, selem)

# 노이즈 제거 효과 비교
noisy = binary.copy()
rng = np.random.default_rng(42)
noise_mask = rng.random(binary.shape) < 0.02
noisy[noise_mask] = ~noisy[noise_mask]  # 노이즈 추가
denoised = morphology.opening(noisy, morphology.disk(2))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
titles = ['원본 이진화', '침식 (erosion)', '팽창 (dilation)',
          '열기 (opening)', '닫기 (closing)', '노이즈 추가',
          '노이즈 제거 (opening)', '원본 이미지']
images = [binary, eroded, dilated, opened, closed, noisy, denoised, img]
cmaps = ['gray'] * 7 + ['gray']
for ax, im, t, cm in zip(axes.flat, images, titles, cmaps):
    ax.imshow(im, cmap=cm)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 erosion: 객체 축소, 작은 노이즈 제거')
print('💡 dilation: 객체 확대, 작은 구멍 채우기')
print('💡 opening: 노이즈 제거에 효과적 (침식 후 팽창)')
print('💡 closing: 구멍 채우기에 효과적 (팽창 후 침식)')
print('💡 morphology.disk(r): 원형 구조 요소, morphology.square(s): 정사각형')

## 4. 노이즈 제거와 복원

scikit-image는 다양한 노이즈 제거 필터를 제공합니다.
가우시안, 중간값, 총변분(TV) 필터 등이 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  노이즈 제거 필터                        │
# │  gaussian, median, total_variation       │
# └─────────────────────────────────────────┘

from skimage import data, filters, restoration, util
from skimage.util import random_noise

# 깨끗한 이미지
img = data.camera()
img_float = img / 255.0

# 노이즈 추가
img_noisy_gauss = random_noise(img_float, mode='gaussian', var=0.01)
img_noisy_sp = random_noise(img_float, mode='s&p', amount=0.05)
img_noisy_poisson = random_noise(img_float, mode='poisson')

# 노이즈 제거
img_denoised_gauss = filters.gaussian(img_noisy_gauss, sigma=1)
img_denoised_median = filters.median(util.img_as_ubyte(img_noisy_sp), morphology.disk(3))
img_denoised_tv = restoration.denoise_tv_chambolle(img_noisy_gauss, weight=0.1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
images = [
    img_float, img_noisy_gauss, img_noisy_sp, img_noisy_poisson,
    img_denoised_gauss, img_denoised_median/255.0, img_denoised_tv, img_float
]
titles = ['원본', '가우시안 노이즈', '소금/후추', '포아송 노이즈',
          '가우시안 필터', '중간값 필터', 'TV 필터', '원본 (비교)']
cmaps = ['gray'] * 8
for ax, im, t, cm in zip(axes.flat, images, titles, cmaps):
    ax.imshow(im, cmap=cm)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 gaussian: 가우시안 노이즈에 효과적, 에지가 둔해짐')
print('💡 median: 소금/후추 노이즈에 가장 효과적')
print('💡 denoise_tv_chambolle: 에지를 보존하면서 노이즈 제거')
print('💡 random_noise(): 다양한 노이즈 모델 (gaussian, s&p, poisson, speckle)')

## 🎯 연습 문제

1. `data.astronaut()` 이미지를 흑백으로 변환하고 Sobel과 Canny 에지를 비교하세요.
2. `data.page()` 이미지에 `threshold_local()`을 적용하고, block_size를 15, 35, 75로 변경해 결과를 비교하세요.
3. 이진화 이미지에 소금/후추 노이즈를 추가하고, opening과 closing을 순서대로 적용해보세요.
4. 가우시안 노이즈 이미지에 대해 `denoise_tv_chambolle`의 weight 매개변수를 0.05, 0.1, 0.2로 변경하며 결과를 비교하세요.
5. `morphology.disk(3)`과 `morphology.square(5)` 구조 요소로 opening을 적용하고 차이를 설명하세요.